In [4]:
# Importing libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#  Loading the Documents
guideline_path = "guideline.pdf"
content_path = "content.pdf"

print("Loading documents...")
loader_guideline = PyPDFLoader(guideline_path)
loader_content = PyPDFLoader(content_path)

guideline_docs = loader_guideline.load()
content_docs = loader_content.load()

print(f"Loaded {len(guideline_docs)} pages of guidelines.")
print(f"Loaded {len(content_docs)} pages of content.")

#  Chunking 
# splitting text into 500 character chunks with a 50 character overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

guideline_chunks = text_splitter.split_documents(guideline_docs)
content_chunks = text_splitter.split_documents(content_docs)

print(f"\n--- SUCCESS ---")
print(f"Guidelines split into {len(guideline_chunks)} chunks.")
print(f"Content split into {len(content_chunks)} chunks.")

# Previewing the chunk 
print("\nSample Content Chunk:")
print(content_chunks[0].page_content)

Loading documents...
Loaded 1 pages of guidelines.
Loaded 3 pages of content.

--- SUCCESS ---
Guidelines split into 1 chunks.
Content split into 5 chunks.

Sample Content Chunk:
PHP CONDITIONAL STATEMENTS 
To perform different actions for different conditions, You can use 
conditional statements in your PHP code to do this. 
 if statement - executes some code if one condition is true
 if...else statement - executes 
some code if a condition is 
true and another code if that condition is false 
 if...elseif....else statement - executes different codes for 
more than two conditions 
 switch statement - selects one of many blocks of code to 
be executed


In [ ]:
# Importing libraries
# HuggingFaceEmbeddings to convert text to numbers (vectors)
# FAISS is the database that stores the vectors for fast searching
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# initialising the embedding model all-MiniLM-L6-v2 
print("Initializing embedding model...")
# converts every chunk of text into a vector
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# creating the vector db
print("Creating FAISS index... (This might take a moment)")
# the text chunks are ran through the model to build a searchable index
vector_db = FAISS.from_documents(content_chunks, embedding_model)

print("Index created successfully!")

#  saving checkpoint (aaysha suggested)
# this index is saved to a folder named faiss_index_react so the folder can be loaded instead of creating vector db step again
vector_db.save_local("faiss_index_react")
print("Checkpoint saved to folder 'faiss_index_react'")

Initializing embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\parallex_project\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shazn\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating FAISS index... (This might take a moment)
Index created successfully!
Checkpoint saved to folder 'faiss_index_react'


In [ ]:
# Loading from checkpoint (Simulating a new session) 
# This proves we can load the data without rebuilding it.
# allow_dangerous_deserialization is required for security, but safe since we made the file.
new_db = FAISS.load_local("faiss_index_react", embedding_model, allow_dangerous_deserialization=True)

#  defining th test query
query = "How do I check multiple conditions?" 

# performing the search
# k=3 gives the the top 3 most relevant chunks
print(f"\nSearching for: '{query}'")
results = new_db.similarity_search(query, k=3)

print("\n--- RESULTS ---")
for i, doc in enumerate(results):
    print(f"\n[Result {i+1}]")
    print(f"Content: {doc.page_content}...") # Printing the matching text
    print("-" * 50)


Searching for: 'How do I check multiple conditions?'

--- SEARCH RESULTS ---

[Result 1]
Content: PHP CONDITIONAL STATEMENTS 
To perform different actions for different conditions, You can use 
conditional statements in your PHP code to do this. 
 if statement - executes some code if one condition is true
 if...else statement - executes 
some code if a condition is 
true and another code if that condition is false 
 if...elseif....else statement - executes different codes for 
more than two conditions 
 switch statement - selects one of many blocks of code to 
be executed...
--------------------------------------------------

[Result 2]
Content: be executed 
 
PHP - THE IF STATEMENT 
The if statement executes some code if one condition is true. 
Syntax 
 if (condition) { 
code to be executed if condition is true; 
} 
<?php 
$t = date("H"); 
if ($t < "20") { 
echo "Have a good day!"; 
} 
?> 
 
The if....else statement executes some code if a condition is 
true and another code if tha